# Montly Order Summary

For each of the customer, produce the following summary per month

1. total orders
2. total item bought
3. total amount spent

In [0]:
df_orders = spark.table('gizmobox.silver.py_orders')
display(df_orders)

In [0]:
from pyspark.sql import functions as F
df_orders_and_month = (
    df_orders
    .withColumn('order_month', F.date_format('transaction_timestamp', 'yyyy-MM'))
)
display(df_orders_and_month)

In [0]:
df_orders_grouped = (
    df_orders_and_month
    .groupBy('order_month', 'customer_id')
    .agg(
        F.countDistinct('order_id').alias('total_orders'),
        F.sum('quantity').alias('total_items_bought'),
        F.sum(F.col('price') * F.col('quantity')).alias('total_amount').alias('total_amount')
    )
)

display(df_orders_grouped.orderBy(F.asc('order_month'), F.asc('customer_id')))

In [0]:
df_orders_grouped.writeTo('gizmobox.gold.py_order_summary_monthly').createOrReplace()

In [0]:
%sql
select * from gizmobox.gold.py_order_summary_monthly